# Kaggle — BTC/ETH multi-task training

> **Decision-support only. Not financial advice.** Markets are noisy and adversarial. A model that fails to beat baselines after fees/slippage must not be used for trading.


## 1. Kaggle setup

In [ ]:
# --- Kaggle setup ---
# Kaggle gives 2x T4 when "GPU T4 x2" accelerator is selected. The project
# auto-detects and uses MirroredStrategy. Add this repo as a Kaggle dataset or
# clone it, then:
import os, sys, subprocess
# If the repo isn't already present, clone it (replace with your fork URL):
# subprocess.run(["git", "clone", "https://github.com/mindees/crypto-ai.git"], check=False)
# os.chdir("crypto-ai")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
print("cwd:", os.getcwd())


## 2. Pull dataset

In [ ]:
# --- pull the latest prebuilt dataset from Kaggle (if configured) ---
# If you publish processed parquet as a Kaggle dataset, attach it to this
# notebook and symlink it into data/. Otherwise the build step below
# regenerates features/labels from raw (requires the raw parquet to be present).
import os
from pathlib import Path
KAGGLE_INPUT = Path("/kaggle/input")
if KAGGLE_INPUT.exists():
    print("Kaggle input datasets:", [p.name for p in KAGGLE_INPUT.iterdir()])
else:
    print("not on Kaggle or no input datasets attached.")


## 3. GPU / environment check

In [ ]:
# --- GPU / environment check ---
import json
from src.utils.hardware import detect_hardware
report = detect_hardware()
print(json.dumps(report, indent=2, default=str))

if report.get("would_use_mirrored_strategy"):
    print("\n>>> 2+ GPUs detected — training will use tf.distribute.MirroredStrategy().")
elif report.get("gpu_count", 0) == 1:
    print("\n>>> single GPU — standard single-device training.")
else:
    print("\n>>> no GPU — CPU smoke only. Enable the accelerator for real training.")


## 4. Configuration

In [ ]:
# --- single visible config block (override here, no need to edit src/) ---
ASSETS       = ["BTCUSDT", "ETHUSDT"]
TIMEFRAMES   = ["1h", "4h"]           # train_default per configs/config.yaml
MARKET       = "spot"
SAMPLE       = False                   # True = last 500 bars (smoke); False = full window
PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 25
BATCH_SIZE    = 256
RESUME        = True                   # resume from latest checkpoint if present
print({"assets": ASSETS, "timeframes": TIMEFRAMES, "market": MARKET,
       "sample": SAMPLE, "phase1": PHASE1_EPOCHS, "phase2": PHASE2_EPOCHS,
       "batch_size": BATCH_SIZE, "resume": RESUME})


## 5. Resume support

In [ ]:
# --- resume support: detect the latest run dir with checkpoints ---
from pathlib import Path
runs = sorted((Path("artifacts") / "runs").glob("*")) if (Path("artifacts") / "runs").exists() else []
resumable = [r for r in runs if (r / "phase1_best.keras").exists() or (r / "model.keras").exists()]
if RESUME and resumable:
    LATEST_RUN = resumable[-1].name
    print(f"resumable run found: {LATEST_RUN}")
    print("training will continue from its best checkpoint where supported.")
else:
    LATEST_RUN = None
    print("no prior checkpoint — training from scratch.")


## 6. Build dataset

In [ ]:
# --- build the sequence-windowed dataset (train-only scaler, purged splits) ---
import subprocess, sys
cmd = [sys.executable, "-m", "src.datasets.build_dataset",
       "--symbols", *ASSETS, "--timeframes", *TIMEFRAMES,
       "--market", MARKET, "--sample", "true" if SAMPLE else "false"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 7. Train

In [ ]:
# --- train (uses MirroredStrategy automatically when 2+ GPUs present) ---
# Kaggle dual-T4 -> MirroredStrategy is auto-selected by src/models/train.py.
import subprocess, sys
cmd = [sys.executable, "-m", "src.models.train",
       "--symbols", *ASSETS, "--timeframe", TIMEFRAMES[0],
       "--epochs", str(PHASE2_EPOCHS), "--batch-size", str(BATCH_SIZE),
       "--sample", "true" if SAMPLE else "false"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 8. Evaluate + backtest + register

In [ ]:
# --- honest evaluation + event-driven backtest vs baselines ---
import subprocess, sys
subprocess.run([sys.executable, "-m", "src.models.evaluate", "--latest",
                "--timeframe", TIMEFRAMES[0], "--sample", "true" if SAMPLE else "false"], check=False)
subprocess.run([sys.executable, "-m", "src.backtest.engine", "--latest",
                "--timeframes", *TIMEFRAMES, "--sample", "true" if SAMPLE else "false"], check=False)
print("Eval + backtest reports written under reports/. "
      "Check whether the model BEATS baselines before trusting it.")


In [ ]:
# --- register the trained run as a candidate; dry-run the promotion gates ---
import subprocess, sys
subprocess.run([sys.executable, "-m", "src.models.registry", "--sync"], check=False)
subprocess.run([sys.executable, "-m", "src.models.registry", "--list"], check=False)
subprocess.run([sys.executable, "-m", "src.models.promote", "--latest", "--dry-run"], check=False)


## 9. Export artifacts

In [ ]:
# --- export artifacts to /kaggle/working (downloadable / dataset-versionable) ---
import shutil
from pathlib import Path
out = Path("/kaggle/working/artifacts") if Path("/kaggle/working").exists() else Path("artifacts_export")
out.mkdir(parents=True, exist_ok=True)
src_runs = Path("artifacts/runs")
if src_runs.exists():
    latest = sorted(src_runs.glob("*"))[-1]
    shutil.copytree(latest, out / latest.name, dirs_exist_ok=True)
    print(f"exported {latest.name} -> {out}")
print("Also copy reports/ for the eval + backtest summaries.")
